# 05-2 — Evaluations

Runs an LLM-judged evaluation suite against a **ReAct agent**.

Architecture:
- `EvalDataset` — a list of `EvalCase` (input + expected output + tags)
- `EvalRunner` — runs the agent for every case with parallelism + timeout
- `LLMJudge` — scores each response against multiple criteria in parallel
- `CostTracker` — hooks into the LLM lifecycle to measure token costs
- Reports exported as **JSON** and **Markdown**

**Prerequisites**: `OPENAI_API_KEY` set and `results/` directory writable.

In [1]:
import os

CHAT_MODEL = os.environ.get("CHAT_MODEL", "openai/gpt-5.4-mini")
API_KEYS = {
    "openai": os.environ.get("OPENAI_API_KEY", ""),
    "anthropic": os.environ.get("ANTHROPIC_API_KEY", ""),
    "google": os.environ.get("GOOGLE_API_KEY", os.environ.get("GEMINI_API_KEY", "")),
    "groq": os.environ.get("GROQ_API_KEY", os.environ.get("GROK_API_KEY", "")),
    "openrouter": os.environ.get("OPENROUTER_API_KEY", ""),
}

os.makedirs("results", exist_ok=True)
print("Results dir ready")
print("CHAT_MODEL:", CHAT_MODEL)

Results dir ready
CHAT_MODEL: openai/gpt-5.4-mini


In [ ]:
from ravi.core.agent_catalog import AgentCatalog
from ravi.core.agents.react_agent import ReActAgent
from ravi.integrations.llm.factory import create_model_client
from ravi.core.tools.builtin_tools import CalculatorTool, GetCurrentTimeTool
from ravi.core.memory.unbounded_memory import UnboundedMemory
from ravi.core.context.implementations import UnboundedContext
from ravi.core.hooks import HookEvent, HookManager, CostTracker
from ravi.evals import (
    EvalCase, EvalDataset, EvalRunner, LLMJudge,
    CORRECTNESS, HELPFULNESS, SAFETY, CONCISENESS,
)

: 

## Define an evaluation dataset

In [3]:
dataset = EvalDataset(
    name='basic_math_qa',
    description='Basic math and tool-use tests',
    cases=[
        EvalCase(input='What is 15 * 7?', expected_output='105', tags=['math']),
        EvalCase(input='Calculate the square root of 144', expected_output='12', tags=['math']),
        EvalCase(input='What is 2^10?', expected_output='1024', tags=['math']),
        EvalCase(input='What is the current time?', expected_output=None, tags=['tools']),
        EvalCase(input='If a train travels 60 mph for 2.5 hours, how far does it go?', expected_output='150 miles', tags=['word_problem']),
    ],
)
print(f'Dataset: {dataset.name} ({dataset.size} cases)')

Dataset: basic_math_qa (5 cases)


## Build agent, judge, and run evaluation

In [ ]:
async def run_eval():
    hooks = HookManager()
    cost_tracker = CostTracker(model=CHAT_MODEL)
    hooks.register(HookEvent.LLM_END, cost_tracker.on_llm_end)
    hooks.register(HookEvent.RUN_END, cost_tracker.on_run_end)

    catalog = AgentCatalog()
    catalog.register_model("primary", create_model_client(CHAT_MODEL, api_keys=API_KEYS))
    catalog.register_memory("default", UnboundedMemory())
    catalog.register_context("default", UnboundedContext())
    catalog.register_tool(CalculatorTool())
    catalog.register_tool(GetCurrentTimeTool())

    agent = ReActAgent(
        name="eval-agent",
        description="Agent under evaluation",
        catalog=catalog,
        hooks=hooks,
        max_iterations=5,
        verbose=False,
    )

    judge = LLMJudge(
        model_client=create_model_client(CHAT_MODEL, api_keys=API_KEYS),
        criteria=[CORRECTNESS, HELPFULNESS, SAFETY, CONCISENESS],
        parallel=True,
    )

    runner = EvalRunner(agent=agent, judge=judge, concurrency=1, case_timeout=60.0, reset_agent=True)

    print(f"Running evaluation with model: {CHAT_MODEL}")
    report = await runner.run(dataset)
    print(report.summary())

    EvalRunner.export_json(report, "results/eval_report.json")
    EvalRunner.export_markdown(report, "results/eval_report.md")
    print("Reports saved to results/")
    print(f"Cost: {cost_tracker.stats}")

await run_eval()